In [ ]:
from datetime import datetime, timedelta
from airflow.decorators import dag, task
import os

# Base directory where Airflow is running inside the container.
# In this project, both the DuckDB database and the staging folder live here.
BASE_PATH = "/opt/airflow"

# Full path to the DuckDB database file.
# If the file does not exist yet, DuckDB will create it when connecting.
DB_PATH = os.path.join(BASE_PATH, "dw.duckdb")

# Directory where the CSV files arrive.
# The pipeline will scan this folder and process only .csv files.
STAGING_DIR = os.path.join(BASE_PATH, "staging")

# Default Airflow arguments for all tasks in this DAG.
default_args = {
    "owner": "data_engineer",                # Logical owner of the workflow
    "retries": 1,                           # If a task fails, retry it once
    "retry_delay": timedelta(minutes=2),    # Wait 2 minutes before retrying
}


@dag(
    dag_id="elt_duckdb_pipeline_V2",         # Name Airflow uses to identify this DAG
    default_args=default_args,
    schedule="@daily",                       # Run once per day
    start_date=datetime(2025, 1, 1),        # First logical execution date
    catchup=False,                           # Do not backfill old pending dates
)
def elt_pipeline():
    """
    ELT pipeline with incremental loading logic.

    General flow:
    1. Prepare control structures and clean current staging table.
    2. Detect new or modified CSV files in staging.
    3. Load only those new/changed files into staging_raw.
    4. Transform staging data and merge it into the historical fact table.
    5. Calculate final metrics for monitoring.

    Important design idea:
    - staging_raw is temporary for each run.
    - fact_finanzas_elt is historical and cumulative.
    - processed_files is the control table that prevents reprocessing the same file.
    """

    @task()
    def preparar_control_y_staging():
        """
        Prepares the technical environment inside DuckDB for this execution.

        What it does:
        - Ensures the control table processed_files exists.
        - Drops staging_raw so each run starts with a clean staging area.

        Why this matters:
        - processed_files tracks which files have already been processed.
        - staging_raw should only contain data for the current execution,
          not data from previous runs.
        """
        import duckdb
        import logging

        # Open a connection to DuckDB.
        conn = duckdb.connect(DB_PATH)

        # Create the control table if it does not exist yet.
        # This table stores metadata about each processed file so the pipeline
        # can detect whether a file is new or has changed.
        conn.execute("""
            CREATE TABLE IF NOT EXISTS processed_files (
                file_name VARCHAR PRIMARY KEY,         -- File name, e.g. finanzas_mes_8.csv
                file_path VARCHAR,                     -- Full path to the file
                file_size_bytes BIGINT,               -- Size in bytes at processing time
                file_modified_at TIMESTAMP,           -- Last modification timestamp of the file
                processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP -- When pipeline processed it
            )
        """)

        # staging_raw is dropped on every run because staging is not historical.
        # It is only an intermediate landing table for the current batch.
        conn.execute("DROP TABLE IF EXISTS staging_raw")

        logging.info("Tabla staging_raw eliminada para la ejecución actual")
        logging.info("Tabla processed_files verificada")

        # Always close the connection when done.
        conn.close()

    @task()
    def cargar_staging_incremental():
        """
        Loads only new or modified CSV files into staging_raw.

        Incremental rule:
        A file is considered already processed if the following three values match
        an existing row in processed_files:
        - file_name
        - file_size_bytes
        - file_modified_at

        If any of those values changed, the file is treated as new/modified
        and will be reprocessed.

        Returns:
        A dictionary with metadata about the load:
        - files_found
        - files_to_process
        - rows_loaded
        - processed_files
        """
        import duckdb
        import logging
        import glob
        import os
        from datetime import datetime

        conn = duckdb.connect(DB_PATH)

        # Find all CSV files inside the staging directory.
        # Example result:
        # ['/opt/airflow/staging/finanzas_mes_1.csv', ..., '/opt/airflow/staging/finanzas_mes_8.csv']
        csv_files = sorted(glob.glob(os.path.join(STAGING_DIR, "*.csv")))

        # If there are no files, we still create an empty staging_raw table.
        # This avoids breaking downstream logic that expects the table to exist.
        if not csv_files:
            logging.info("No se encontraron archivos CSV en staging")

            # This creates an empty table with a dummy column.
            # It is only a placeholder because no transformation will happen anyway.
            conn.execute("CREATE TABLE staging_raw AS SELECT * FROM (SELECT 1 AS dummy) WHERE 1=0")
            conn.close()

            return {
                "files_found": 0,
                "files_to_process": 0,
                "rows_loaded": 0,
                "processed_files": []
            }

        # This list will store only the files that need to be processed in this run.
        files_to_process = []

        # Iterate through every CSV file found in staging.
        for file_path in csv_files:
            # Read file metadata from the operating system.
            stat = os.stat(file_path)
            file_name = os.path.basename(file_path)
            file_size_bytes = stat.st_size
            file_modified_at = datetime.fromtimestamp(stat.st_mtime)

            # Check if this exact file version was already processed.
            # Parameterized query is safer than string concatenation.
            exists = conn.execute(
                """
                SELECT 1
                FROM processed_files
                WHERE file_name = ?
                  AND file_size_bytes = ?
                  AND file_modified_at = ?
                LIMIT 1
                """,
                [file_name, file_size_bytes, file_modified_at],
            ).fetchone()

            # If no matching record is found, the file is new or changed.
            if exists is None:
                files_to_process.append(
                    {
                        "file_name": file_name,
                        "file_path": file_path,
                        "file_size_bytes": file_size_bytes,
                        "file_modified_at": file_modified_at,
                    }
                )

        # If all files were already processed and unchanged, do not load anything.
        if not files_to_process:
            logging.info("Todos los archivos ya fueron procesados previamente. No se cargará información nueva.")

            # Again, create an empty staging_raw placeholder.
            conn.execute("CREATE TABLE staging_raw AS SELECT * FROM (SELECT 1 AS dummy) WHERE 1=0")
            conn.close()

            return {
                "files_found": len(csv_files),
                "files_to_process": 0,
                "rows_loaded": 0,
                "processed_files": []
            }

        # Build one SELECT statement per file, then union them all together.
        # Each SELECT reads the CSV and adds a source_file column to keep track
        # of where each row came from.
        select_statements = []

        for file_info in files_to_process:
            # Escape single quotes to avoid SQL breaking if a path/name contains quotes.
            safe_path = file_info["file_path"].replace("'", "''")
            safe_name = file_info["file_name"].replace("'", "''")

            select_statements.append(
                f"""
                SELECT
                    *,                              -- Bring all columns from the CSV
                    '{safe_name}' AS source_file   -- Add the file name as traceability metadata
                FROM read_csv_auto('{safe_path}')
                """
            )

        # Combine all file reads into a single SQL statement.
        # Example:
        # SELECT * FROM file1
        # UNION ALL
        # SELECT * FROM file2
        union_sql = "\nUNION ALL\n".join(select_statements)

        # Create staging_raw with the union of all new/changed files.
        conn.execute(f"CREATE TABLE staging_raw AS {union_sql}")

        # Count how many rows were loaded into staging.
        rows_loaded = conn.execute("SELECT COUNT(*) FROM staging_raw").fetchone()[0]

        # Register each processed file in the control table.
        # This prevents reprocessing the same exact version later.
        for file_info in files_to_process:
            conn.execute(
                """
                INSERT INTO processed_files (
                    file_name,
                    file_path,
                    file_size_bytes,
                    file_modified_at,
                    processed_at
                ) VALUES (?, ?, ?, ?, CURRENT_TIMESTAMP)
                """,
                [
                    file_info["file_name"],
                    file_info["file_path"],
                    file_info["file_size_bytes"],
                    file_info["file_modified_at"],
                ],
            )

        logging.info(f"Archivos encontrados: {len(csv_files)}")
        logging.info(f"Archivos nuevos o modificados a procesar: {len(files_to_process)}")
        logging.info(f"Registros cargados en staging_raw: {rows_loaded}")
        logging.info(
            "Archivos procesados en esta ejecución: %s",
            ", ".join([f["file_name"] for f in files_to_process])
        )

        conn.close()

        # Return metadata for the next task.
        return {
            "files_found": len(csv_files),
            "files_to_process": len(files_to_process),
            "rows_loaded": rows_loaded,
            "processed_files": [f["file_name"] for f in files_to_process],
        }

    @task()
    def transformar_datos(staging_info: dict):
        """
        Transforms data from staging_raw and merges it into the historical fact table.

        Key ideas:
        - If there are no new files, do nothing.
        - fact_finanzas_elt is cumulative: it stores historical results.
        - New rows are inserted.
        - Existing rows are updated if their values changed.

        Business key used for matching historical rows:
        - id
        - fecha
        """
        import duckdb
        import logging

        conn = duckdb.connect(DB_PATH)

        # If the previous task found no new files, skip transformation.
        if staging_info.get("files_to_process", 0) == 0:
            logging.info("No hay archivos nuevos para transformar. Se conserva el histórico sin cambios.")
            conn.close()
            return

        # Count how many rows arrived in staging.
        total_staging = conn.execute("SELECT COUNT(*) FROM staging_raw").fetchone()[0]

        # Create the historical fact table if it does not exist yet.
        # This table stores the transformed and cumulative financial data.
        conn.execute("""
            CREATE TABLE IF NOT EXISTS fact_finanzas_elt (
                id BIGINT,                                   -- Business identifier
                salario DOUBLE,                              -- Salary value
                gastos DOUBLE,                               -- Expenses value
                fecha DATE,                                  -- Normalized date
                correo_hash VARCHAR,                         -- Hashed email for privacy
                utilidad DOUBLE,                             -- Derived metric: salario - gastos
                source_file VARCHAR,                         -- Origin file of the row
                fecha_carga TIMESTAMP DEFAULT CURRENT_TIMESTAMP,        -- First load timestamp
                fecha_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP -- Last update timestamp
            )
        """)

        # Remove temporary transformation table if it exists from an earlier failed session.
        conn.execute("DROP TABLE IF EXISTS tmp_fact_finanzas_elt")

        # Build a temporary table with the transformed version of the current staging data.
        conn.execute("""
            CREATE TEMP TABLE tmp_fact_finanzas_elt AS
            WITH base AS (
                SELECT
                    id,

                    -- Keep salary as-is.
                    salario,

                    -- Replace NULL expenses with 0 so calculations do not break.
                    COALESCE(gastos, 0) AS gastos,

                    -- Normalize fecha into DATE format.
                    -- The code supports three input patterns:
                    -- 1. YYYY-MM-DD
                    -- 2. DD/MM/YYYY
                    -- 3. MM-DD-YYYY
                    CASE
                        WHEN regexp_matches(CAST(fecha AS VARCHAR), '^[0-9]{4}-[0-9]{2}-[0-9]{2}$')
                            THEN CAST(CAST(fecha AS VARCHAR) AS DATE)
                        WHEN regexp_matches(CAST(fecha AS VARCHAR), '^[0-9]{2}/[0-9]{2}/[0-9]{4}$')
                            THEN CAST(STRPTIME(CAST(fecha AS VARCHAR), '%d/%m/%Y') AS DATE)
                        WHEN regexp_matches(CAST(fecha AS VARCHAR), '^[0-9]{2}-[0-9]{2}-[0-9]{4}$')
                            THEN CAST(STRPTIME(CAST(fecha AS VARCHAR), '%m-%d-%Y') AS DATE)
                        ELSE NULL
                    END AS fecha,

                    -- Hash email for privacy/security.
                    -- Original email is not stored in the fact table.
                    CASE
                        WHEN correo IS NOT NULL THEN sha256(CAST(correo AS VARCHAR))
                        ELSE NULL
                    END AS correo_hash,

                    source_file,

                    -- Timestamp for this transformation event.
                    CURRENT_TIMESTAMP AS fecha_proceso
                FROM staging_raw
            ),

            filtrado AS (
                SELECT
                    id,
                    salario,
                    gastos,
                    fecha,
                    correo_hash,

                    -- Derived business metric.
                    salario - gastos AS utilidad,

                    source_file,

                    -- fecha_carga and fecha_actualizacion start with the same value
                    -- when the row is first transformed in this run.
                    fecha_proceso AS fecha_carga,
                    fecha_proceso AS fecha_actualizacion
                FROM base
                WHERE
                    -- Data quality filters:
                    id IS NOT NULL
                    AND fecha IS NOT NULL
                    AND salario > 0
                    AND gastos >= 0
            ),

            deduplicado AS (
                SELECT *
                FROM (
                    SELECT
                        *,

                        -- If duplicate rows exist in the current batch for the same
                        -- business key (id, fecha), keep only one row.
                        ROW_NUMBER() OVER (
                            PARTITION BY id, fecha
                            ORDER BY fecha_actualizacion DESC, source_file DESC
                        ) AS rn
                    FROM filtrado
                )
                WHERE rn = 1
            )

            -- Final projection for the temp table.
            SELECT
                id,
                salario,
                gastos,
                fecha,
                correo_hash,
                utilidad,
                source_file,
                fecha_carga,
                fecha_actualizacion
            FROM deduplicado
        """)

        # Number of valid transformed rows from the current staging load.
        total_transformados = conn.execute(
            "SELECT COUNT(*) FROM tmp_fact_finanzas_elt"
        ).fetchone()[0]

        # Count rows that do not exist yet in the historical table.
        # These will be inserted.
        nuevos = conn.execute("""
            SELECT COUNT(*)
            FROM tmp_fact_finanzas_elt s
            LEFT JOIN fact_finanzas_elt t
              ON t.id = s.id
             AND t.fecha = s.fecha
            WHERE t.id IS NULL
        """).fetchone()[0]

        # Count rows that already exist but changed in one or more tracked attributes.
        # These will be updated.
        actualizaciones = conn.execute("""
            SELECT COUNT(*)
            FROM tmp_fact_finanzas_elt s
            JOIN fact_finanzas_elt t
              ON t.id = s.id
             AND t.fecha = s.fecha
            WHERE
                COALESCE(t.salario, -1) <> COALESCE(s.salario, -1)
                OR COALESCE(t.gastos, -1) <> COALESCE(s.gastos, -1)
                OR COALESCE(t.correo_hash, '') <> COALESCE(s.correo_hash, '')
                OR COALESCE(t.utilidad, -1) <> COALESCE(s.utilidad, -1)
                OR COALESCE(t.source_file, '') <> COALESCE(s.source_file, '')
        """).fetchone()[0]

        # Update existing historical rows only when a relevant value changed.
        conn.execute("""
            UPDATE fact_finanzas_elt AS target
            SET
                salario = source.salario,
                gastos = source.gastos,
                correo_hash = source.correo_hash,
                utilidad = source.utilidad,
                source_file = source.source_file,
                fecha_actualizacion = CURRENT_TIMESTAMP
            FROM tmp_fact_finanzas_elt AS source
            WHERE target.id = source.id
              AND target.fecha = source.fecha
              AND (
                    COALESCE(target.salario, -1) <> COALESCE(source.salario, -1)
                    OR COALESCE(target.gastos, -1) <> COALESCE(source.gastos, -1)
                    OR COALESCE(target.correo_hash, '') <> COALESCE(source.correo_hash, '')
                    OR COALESCE(target.utilidad, -1) <> COALESCE(source.utilidad, -1)
                    OR COALESCE(target.source_file, '') <> COALESCE(source.source_file, '')
              )
        """)

        # Insert rows that do not exist yet in the historical table.
        conn.execute("""
            INSERT INTO fact_finanzas_elt (
                id,
                salario,
                gastos,
                fecha,
                correo_hash,
                utilidad,
                source_file,
                fecha_carga,
                fecha_actualizacion
            )
            SELECT
                s.id,
                s.salario,
                s.gastos,
                s.fecha,
                s.correo_hash,
                s.utilidad,
                s.source_file,
                s.fecha_carga,
                s.fecha_actualizacion
            FROM tmp_fact_finanzas_elt s
            LEFT JOIN fact_finanzas_elt t
              ON t.id = s.id
             AND t.fecha = s.fecha
            WHERE t.id IS NULL
        """)

        # Final row count in the historical fact table after merge.
        total_final = conn.execute(
            "SELECT COUNT(*) FROM fact_finanzas_elt"
        ).fetchone()[0]

        logging.info(f"Registros leídos en staging: {total_staging}")
        logging.info(f"Registros válidos transformados: {total_transformados}")
        logging.info(f"Registros nuevos insertados: {nuevos}")
        logging.info(f"Registros actualizados: {actualizaciones}")
        logging.info(f"Total histórico en fact_finanzas_elt: {total_final}")
        logging.info("Carga incremental completada sin eliminar histórico")

        conn.close()

    @task()
    def metricas_finales():
        """
        Calculates a simple summary of the historical fact table.

        This task is mainly for monitoring and validation.
        It helps verify that the table is populated and that the derived metric
        utilidad behaves as expected.
        """
        import duckdb
        import logging

        conn = duckdb.connect(DB_PATH)

        # Simple aggregate metrics over the cumulative historical table.
        resumen = conn.execute("""
            SELECT
                COUNT(*) AS total,
                AVG(utilidad) AS promedio_utilidad,
                MAX(utilidad) AS max_utilidad
            FROM fact_finanzas_elt
        """).fetchdf()

        # Print the summary into Airflow logs.
        logging.info(resumen.to_string(index=False))

        conn.close()

    # Instantiate each task.
    preparar = preparar_control_y_staging()
    staging_info = cargar_staging_incremental()
    transformacion = transformar_datos(staging_info)
    metricas = metricas_finales()

    # Define task execution order.
    # 1. prepare environment
    # 2. load new/changed files into staging
    # 3. transform and merge into fact table
    # 4. compute monitoring metrics
    preparar >> staging_info >> transformacion >> metricas


# Build the DAG object so Airflow can discover it.
elt_dag = elt_pipeline()